<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 3: Probabilidad y distribuciones de probabilidad </span><br/>
  <span style="color:DodgerBlue; font-size:140%;"> Probabilidad marginal, condicional y conjunta</span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 3: Probabilidad: Aplicaciones </font>**

# 1. Bibliotecas



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


---
# Concepto de probabilidad: clásica y frecuentista

## ¿Qué es la probabilidad?

La **probabilidad** es un número entre 0 y 1 (o entre 0% y 100%) que mide qué tan posible es que ocurra un evento. Un valor de 0 significa que el evento es imposible; un valor de 1, que es seguro. Existen dos formas principales de asignar (o estimar) ese número:

- **Probabilidad clásica (regla de Laplace):** se usa cuando conocemos de antemano todos los resultados posibles de un experimento y asumimos que todos son **igualmente probables**.
- **Probabilidad frecuentista:** se usa cuando estimamos la probabilidad a partir de la **frecuencia relativa observada** al repetir un experimento (o al observar muchos datos reales) — es el enfoque que más usamos en ciencia, porque rara vez conocemos de antemano un modelo "perfecto".

## Probabilidad clásica (regla de Laplace)

$$P(A) = \frac{\text{casos favorables de } A}{\text{casos totales posibles}}$$

### Nuestro escenario: una caja de muestras minerales

Imagina una caja de laboratorio con **50 muestras minerales**, clasificadas por tipo:

In [ ]:
caja_minerales = pd.DataFrame({
    "tipo": ["cuarzo", "calcita", "pirita", "magnetita"],
    "cantidad": [15, 10, 15, 10]
})
caja_minerales["probabilidad_clasica"] = caja_minerales["cantidad"] / caja_minerales["cantidad"].sum()

caja_minerales

Si sacamos **una muestra al azar** de la caja, cada una de las 50 muestras tiene la misma probabilidad de ser elegida, por eso podemos usar la regla de Laplace.


In [ ]:
total_muestras = caja_minerales["cantidad"].sum()

#CAlcularemos la probabilidad de sacar un cuarzo
p_cuarzo = caja_minerales.loc[caja_minerales["tipo"] == "cuarzo", "cantidad"].values[0] / total_muestras
print(f"P(cuarzo) = {p_cuarzo:.2f}")

#Ahora la probabilidad de no sacar una pirita
# "No es pirita" = todas las que NO son pirita
favorables_no_pirita = total_muestras - caja_minerales.loc[caja_minerales["tipo"] == "pirita", "cantidad"].values[0]
p_no_pirita = favorables_no_pirita / total_muestras
print(f"P(no es pirita) = {p_no_pirita:.2f}")

## Ejercicio:

Usando `caja_minerales`:

1. Calcula `P(calcita)`.
2. Calcula `P(magnetita o pirita)` — recuerda que un mineral no puede ser de dos tipos a la vez, así que puedes sumar directamente los casos favorables.

## Probabilidad frecuentista

En la vida real, pocas veces tenemos un modelo "perfecto" como el de la caja. En cambio, **estimamos** la probabilidad de un evento observando qué tan seguido ocurre al repetir el experimento muchas veces:

$$P(A) \approx \frac{\text{número de veces que ocurrió } A}{\text{número total de repeticiones}}$$

Un resultado fundamental de la estadística —la **Ley de los Grandes Números**— nos dice que, si el experimento se repite *suficientes* veces, la frecuencia relativa observada se acerca cada vez más a la probabilidad clásica "verdadera" (cuando esta existe).

### Viendo la Ley de los Grandes Números en acción

Vamos a simular que sacamos una muestra de la caja (con reemplazo, como si repusiéramos cada muestra antes de la siguiente extracción) **10,000 veces**, y vamos a rastrear la frecuencia relativa de "cuarzo" conforme aumenta el número de extracciones.

In [ ]:
np.random.seed(11)

tipos = caja_minerales["tipo"].tolist()
probabilidades = caja_minerales["probabilidad_clasica"].tolist()

N = 10_000
extracciones = np.random.choice(tipos, size=N, p=probabilidades)

es_cuarzo = (extracciones == "cuarzo").astype(int)
frecuencia_acumulada = np.cumsum(es_cuarzo) / np.arange(1, N + 1)

print(f"Frecuencia relativa tras 10 extracciones:    {frecuencia_acumulada[9]:.3f}")
print(f"Frecuencia relativa tras 1,000 extracciones:  {frecuencia_acumulada[999]:.3f}")
print(f"Frecuencia relativa tras 10,000 extracciones: {frecuencia_acumulada[-1]:.3f}")
print(f"Probabilidad clásica (verdadera): {p_cuarzo:.3f}")

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(np.arange(1, N + 1), frecuencia_acumulada, color="steelblue", linewidth=1)
plt.axhline(p_cuarzo, color="crimson", linestyle="--", label=f"Probabilidad clásica = {p_cuarzo:.2f}")
plt.xscale("log")
plt.xlabel("Número de extracciones (escala logarítmica)")
plt.ylabel("Frecuencia relativa acumulada de 'cuarzo'")
plt.title("Ley de los Grandes Números: la frecuencia relativa converge a la probabilidad clásica")
plt.legend()
plt.show()

Al principio (pocas extracciones) la frecuencia relativa "salta" bastante — tuvimos suerte o mala suerte. Pero conforme aumentan las repeticiones, se estabiliza cada vez más cerca de la probabilidad clásica real (0.30). **Esto es exactamente lo que vimos con el muestreo repetido** — el mismo principio que allá llamamos "variabilidad muestral" es, en el fondo, la Ley de los Grandes Números.

### Ejercicio:

1. Repite la simulación anterior (puedes usar `np.random.seed(22)` para tener resultados reproducibles), pero ahora rastrea la frecuencia relativa de **"pirita"**.
2. Grafica la frecuencia relativa acumulada contra el número de extracciones (en escala logarítmica), con una línea horizontal en la probabilidad clásica de pirita.
3. Imprime la frecuencia relativa final (tras 10,000 extracciones) y compárala con la probabilidad clásica.

---
# Propiedades y axiomas de la probabilidad

## Los axiomas de la probabilidad

Toda probabilidad, sin importar cómo se calcule (clásica, frecuentista, u otro enfoque), debe cumplir tres reglas básicas, llamadas **axiomas**:

1. **No negatividad:** $P(A) \geq 0$ para cualquier evento $A$.
2. **Normalización:** $P(S) = 1$, donde $S$ es el espacio muestral completo (todos los resultados posibles).
3. **Aditividad para eventos mutuamente excluyentes:** si $A$ y $B$ no pueden ocurrir al mismo tiempo, entonces $P(A \cup B) = P(A) + P(B)$.

De estos tres axiomas se derivan varias propiedades muy útiles en la práctica.

### Verificando los axiomas con la caja de minerales

In [ ]:
# Axioma 1: ninguna probabilidad puede ser negativa
print("¿Todas las probabilidades son >= 0?", (caja_minerales["probabilidad_clasica"] >= 0).all())


In [ ]:
# Axioma 2: la suma de las probabilidades de TODOS los resultados posibles debe ser 1
suma_total = caja_minerales["probabilidad_clasica"].sum()
print(f"Suma de todas las probabilidades: {suma_total:.4f}  (debe ser 1.0)")



In [ ]:
# Axioma 3: cuarzo y pirita son mutuamente excluyentes (una muestra no puede ser ambas a la vez)
p_cuarzo_or_pirita_axioma = p_cuarzo + caja_minerales.loc[caja_minerales["tipo"]=="pirita","probabilidad_clasica"].values[0]
# Verificación directa contando casos favorables
favorables_directo = caja_minerales.loc[caja_minerales["tipo"].isin(["cuarzo","pirita"]), "cantidad"].sum()
p_cuarzo_or_pirita_directo = favorables_directo / total_muestras

print(f"P(cuarzo) + P(pirita) = {p_cuarzo_or_pirita_axioma:.3f}")
print(f"P(cuarzo o pirita) contado directamente = {p_cuarzo_or_pirita_directo:.3f}")

## Propiedades derivadas

**Regla del complemento:** $P(A^c) = 1 - P(A)$, donde $A^c$ ("A complemento") significa "que NO ocurra A".

**Regla de la suma general** (para eventos que **sí** pueden ocurrir a la vez, es decir, no mutuamente excluyentes):

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

Restamos $P(A \cap B)$ porque, si no lo hiciéramos, estaríamos contando dos veces los casos en los que ambos eventos ocurren a la vez.

### Regla del complemento

Ya calculamos `P(no es pirita)` sumando los casos directamente. Comprobemos que la regla del complemento da el mismo resultado:

In [ ]:
p_pirita = caja_minerales.loc[caja_minerales["tipo"] == "pirita", "probabilidad_clasica"].values[0]
p_no_pirita_complemento = 1 - p_pirita

print(f"P(pirita) = {p_pirita:.2f}")
print(f"P(no pirita) con la regla del complemento = {p_no_pirita_complemento:.2f}")
print(f"P(no pirita) calculada antes directamente = {p_no_pirita:.2f}")

### Agregando un segundo atributo: dureza

Para usar la regla de la suma general necesitamos dos eventos que **sí puedan ocurrir a la vez**. Agreguemos un segundo atributo a cada muestra de la caja: su **dureza** (alta o baja, según la escala de Mohs). Una muestra puede ser, por ejemplo, cuarzo **y** tener dureza alta al mismo tiempo.

In [ ]:
minerales_detalle = pd.DataFrame({
    "tipo": (["cuarzo"] * 15) + (["calcita"] * 10) + (["pirita"] * 15) + (["magnetita"] * 10),
    "dureza": (
        ["alta"] * 12 + ["baja"] * 3 +      # cuarzo: 12 alta, 3 baja
        ["alta"] * 2 + ["baja"] * 8 +       # calcita: 2 alta, 8 baja
        ["alta"] * 9 + ["baja"] * 6 +       # pirita: 9 alta, 6 baja
        ["alta"] * 6 + ["baja"] * 4         # magnetita: 6 alta, 4 baja
    )
})

print("Total de muestras:", len(minerales_detalle))
minerales_detalle.groupby(["tipo", "dureza"]).size()

### Regla de la suma general

Calculemos $P(\text{cuarzo} \cup \text{dureza alta})$, es decir, la probabilidad de que una muestra al azar sea cuarzo **o** tenga dureza alta (o ambas cosas).

In [ ]:
n = len(minerales_detalle)

p_cuarzo2 = (minerales_detalle["tipo"] == "cuarzo").mean()
p_dureza_alta = (minerales_detalle["dureza"] == "alta").mean()
p_cuarzo_y_dureza_alta = ((minerales_detalle["tipo"] == "cuarzo") & (minerales_detalle["dureza"] == "alta")).mean()

p_union_formula = p_cuarzo2 + p_dureza_alta - p_cuarzo_y_dureza_alta
print(f"P(cuarzo) = {p_cuarzo2:.2f}")
print(f"P(dureza alta) = {p_dureza_alta:.2f}")
print(f"P(cuarzo y dureza alta) = {p_cuarzo_y_dureza_alta:.2f}")
print(f"P(cuarzo o dureza alta) [fórmula] = {p_union_formula:.2f}")

# Verificación: contar directamente las muestras que cumplen al menos una condición
cumple_directo = ((minerales_detalle["tipo"] == "cuarzo") | (minerales_detalle["dureza"] == "alta")).mean()
print(f"P(cuarzo o dureza alta) [conteo directo] = {cumple_directo:.2f}")

### Ejercicio

Usando `minerales_detalle`, calcula $P(\text{pirita} \cup \text{dureza baja})$ de dos formas:

1. Con la fórmula de la suma general: $P(A)+P(B)-P(A\cap B)$.
2. Contando directamente las muestras que cumplen al menos una de las dos condiciones.

Confirma que ambos resultados coinciden.

---
# Probabilidad condicional, marginal y conjunta

Cuando tenemos **dos variables categóricas** medidas sobre los mismos elementos (como `tipo` y `dureza` en nuestra caja de minerales), podemos organizar los conteos en una **tabla de contingencia**, y a partir de ella calcular tres tipos de probabilidad:

| Tipo | Significado | Cómo se calcula |
|---|---|---|
| **Conjunta** — $P(A \cap B)$ | Probabilidad de que ocurran A y B a la vez | (conteo de la celda) / (total general) |
| **Marginal** — $P(A)$ | Probabilidad de A, sin importar qué pase con B | (total de la fila o columna de A) / (total general) |
| **Condicional** — $P(A \mid B)$ | Probabilidad de A, *sabiendo que* B ya ocurrió | $P(A \cap B) / P(B)$ |


### Tabla de contingencia con la caja de minerales

In [ ]:
tabla_minerales = pd.crosstab(minerales_detalle["tipo"], minerales_detalle["dureza"])
tabla_minerales["total"] = tabla_minerales.sum(axis=1)
tabla_minerales.loc["total"] = tabla_minerales.sum()
tabla_minerales

In [ ]:
n_total_min = len(minerales_detalle)

# Probabilidades MARGINALES
p_marginal_tipo = minerales_detalle["tipo"].value_counts(normalize=True)
p_marginal_dureza = minerales_detalle["dureza"].value_counts(normalize=True)
print("--- Marginales: tipo ---")
print(p_marginal_tipo)
print("\n--- Marginales: dureza ---")
print(p_marginal_dureza)

In [ ]:
# Probabilidad CONJUNTA de cada combinación (tabla completa / N)
tabla_conjunta = pd.crosstab(minerales_detalle["tipo"], minerales_detalle["dureza"], normalize="all")
print("--- Probabilidades conjuntas P(tipo ∩ dureza) ---")
tabla_conjunta

In [ ]:
# Probabilidad CONDICIONAL: P(dureza | tipo) -- normalizamos cada FILA para que sume 1
tabla_condicional_dureza_dado_tipo = pd.crosstab(minerales_detalle["tipo"], minerales_detalle["dureza"], normalize="index")
print("--- P(dureza | tipo) ---")
tabla_condicional_dureza_dado_tipo

Observa, por ejemplo, que $P(\text{dureza alta} \mid \text{cuarzo}) = 0.80$, mucho más alta que la probabilidad marginal $P(\text{dureza alta}) = 0.58$. Esto ya nos sugiere que `tipo` y `dureza` **no son independientes** — volveremos a esto en un momento.

### Ejercicio

1. Calcula $P(\text{pirita} \mid \text{dureza baja})$, es decir, la probabilidad de que una muestra sea pirita **dado que** ya sabemos que tiene dureza baja. *Pista: necesitas normalizar por columna en vez de por fila — revisa el argumento `normalize` de `pd.crosstab`.*
2. Compara ese valor con la probabilidad marginal $P(\text{pirita})$. ¿Son parecidos o muy distintos?

# Emisiones de CO2 del Volcán Mauna Loa

Aahora aplicaremos los conocimientos de **Probabilidad** aprendidos en clase a una base de datos de emisiones de CO2 del volcán Manua Loa para estudiar su comportamiento.

Nuestro objetivo es repasar el análisis de datos por medio de la **Estadística descriptiva** y proporcionar información sobre la probabilidad de que ocurran eventos relacionados.





Los datos provienen del Observatorio de Mauna Loa, ubicado en la isla de Hawái. Este observatorio proporciona información sobre las concentraciones de CO2 en la atmósfera desde 1958 y pueden ser descargados en el siguiente enlace: https://gml.noaa.gov/ccgg/trends/data.html

*   Contenido:  La base de datos incluye mediciones mensuales de la concentración de CO2 en partes por millón (ppm).

# Carga de datos

En ocasiones, trabajaremos con bases de datos de terceros. Una forma de cargar estos datos es la siguiente:

In [ ]:
# Proporcionamos el url del archivo que queremos descargar
url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
# Cargamos los datos como lo hemos hecho anteriormente y lo almacenamos en un Data Frame
data = pd.read_csv(url, skiprows=40)  # Saltar las primeras filas que contienen metadatos

# Mostrar las primeras filas del dataframe
data.head()


In [ ]:
# Renombrar las columnas para mayor claridad
data.columns = ["Año", "Mes", "Fecha Decimal", "Promedio", "Corrección", "Num_dias","Desv_stand","Incertidumbre"]
data.head()

In [ ]:

# Graficar las emisiones de CO2
plt.figure(figsize=(10, 6))
plt.plot(data["Fecha Decimal"], data["Promedio"], label="Emisiones de CO2")
plt.grid(True)
plt.xlabel("Fecha")
plt.ylabel("Emisiones de CO2 (ppm)")
plt.title("Emisiones de CO2 del Volcán Mauna Loa")
plt.legend()
plt.show()


# Análisis exploratorio de datos

Queremos estudiar el comportamiento anual de las emisiones de CO2.

In [ ]:
#Definimos las columnas de interés
columnas = ["Año", "Mes", "Promedio"]

# Creamos un nuevo Data Frame sólo con las columnas que usaremos
df_co2 = data[columnas].copy()

# Eliminamos los valores faltantes
df_co2.dropna(inplace=True)
df_co2.head()

## De acuerdo a los datos históricos, ¿En qué mes la concentración de CO2 es mayor?   

In [ ]:
# Crear un boxplot para cada año
plt.figure(figsize=(12, 8))
data.boxplot(column="Promedio", by="Mes", grid=False, rot=90)
plt.title("Boxplot de las Concentraciones Mensuales de CO2 por Año")
plt.suptitle("")  # Eliminar el título automático de pandas
plt.xlabel("Meses")
plt.ylabel("Concentración de CO2 (ppm)")
plt.grid(True)
plt.show()

## De acuerdo a los datos históricos ¿Cuál es la probabilidad de que en un mes cualquiera las emisiones sean mayores a 400 ppm?

In [ ]:
# Filtramos los datos para obtener el número de meses cuyas emisiones son mayores a 400 ppm
ppm_400 = df_co2[df_co2["Promedio"] > 400]
ppm_400


In [ ]:
# Ahora contamos los meses cuyas emisiones son mayores a 400 ppm
meses_400 = len(ppm_400)
meses_400

In [ ]:
# Y también necesitamos los meses totales
meses_totales = len(df_co2)
meses_totales

In [ ]:
# Calculamos la probabilidad (frecuentista) P = numero de ocurrencias / numero de meses totales
probabilidad = meses_400 / meses_totales
probabilidad

## En el scatterplot observamos que las emisiones han ido aumentando (y disminuyendo) con el paso de los años. La probabilidad frecuentista no necesarimente describe la probabilidad de exceder una cantidad de CO2 algún mes del año.

## Necesitamos otra medida de la probabilidad que considere más que la probabilidad marginal. Es decir:


## El concepto de Probabilidad Condicional

## Ejercicios:


1.   Calcular la probabilidad de que las emisiones sean mayores a 400 ppm dado que un mes antes fueron mayores a 390
2.   Calcular la probabilidad de que las emisiones disminuyan dado que fueron mayores a 410 ppm
3.   Calcular la probabilidad de que las emisiones sean mayores a 420 ppm dado que un mes antes fueron mayores a 410
4.   Calcular la probabilidad de que las emisiones disminuyan dado que fueron mayores a 420 ppm

In [ ]:
# Crear una nueva columna para las emisiones del mes anterior
df_co2["promedio mes anterior"] = df_co2["Promedio"].shift(1)
df_co2.head()

In [ ]:
# Filtrar los datos para los meses donde las emisiones del mes anterior fueron mayores a 390 ppm
ppm_390 = df_co2[df_co2["promedio mes anterior"] > 390]

# Calcular la probabilidad marginal
probabilidad_marginal_ppm_390 = len(ppm_390) / len(df_co2)
probabilidad_marginal_ppm_390

In [ ]:
# Calculamos la probabilidad de que las emisiones sean mayores a 390 ppm dado que después son mayores a 400

# Podemos usar ahora la definición de probabilidad conjunta
# contamos los meses en los que que las emisiones fueron mayores a 400 y un mes antes mayores a 390

meses_400_y_390 = df_co2[(df_co2["Promedio"] > 400) & (df_co2["promedio mes anterior"] > 390)]
meses_400_y_390


In [ ]:
meses_400_y_390 = df_co2[(df_co2["Promedio"] > 400) & (df_co2["promedio mes anterior"] > 390)]
#Obtenemos la probabilidad condicional
probabilidad_condicional = len(meses_400_y_390) / len(ppm_390)
probabilidad_condicional

## Ahora es su turno:

2.   Calcular la probabilidad de que las emisiones disminuyan dado que fueron mayores a 410 ppm
3.   Calcular la probabilidad de que las emisiones sean mayores a 420 ppm dado que un mes antes fueron mayores a 410
4.   Calcular la probabilidad de que las emisiones disminuyan dado que fueron mayores a 420 ppm